# Atelier : Classification supervisée d'images (base DB2C : Voiture vs Bateau)
**Pr. Abdelouahed Sabri**

Placez le dossier **DB2C** (contenant `obj_car` et `obj_ship`) à côté de ce notebook, puis exécutez les cellules **dans l'ordre**.
Après chaque cellule, faites une **capture d'écran** de la sortie (elle est numérotée « CAPTURE n » dans le rapport). Les figures sont aussi sauvegardées dans `captures/`.

In [ ]:
%pip install -q opencv-python-headless scikit-image scikit-learn seaborn pandas matplotlib joblib gradio

## Fichier utilitaire : extraction des caractéristiques
Cette cellule crée `features.py`. Le même fichier sera utilisé par la page web, pour garantir des caractéristiques identiques.

In [ ]:
%%writefile features.py
"""
features.py - Extraction des caracteristiques d'une image RGB (IMG_SIZE x IMG_SIZE).
Ce meme fichier est utilise par le notebook (entrainement) ET par la page web (prediction),
afin que les caracteristiques soient calculees exactement de la meme facon.
"""
import numpy as np
import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

CLASSES = ["Voiture", "Bateau"]  # etiquette 0 = Voiture (dossier obj_car), 1 = Bateau (dossier obj_ship)
IMG_SIZE = 32                    # toutes les images sont redimensionnees en IMG_SIZE x IMG_SIZE


def _to_uint8(img):
    """Accepte une image float normalisee [0,1] ou uint8 [0,255] et renvoie du uint8."""
    if img.dtype != np.uint8:
        img = np.clip(np.round(img * 255.0), 0, 255).astype(np.uint8)
    return img


def rgb_stats(img):
    """Moyenne et ecart-type de chaque canal R, G, B -> 6 valeurs."""
    f = img.astype(np.float32)
    return np.concatenate([f.mean(axis=(0, 1)), f.std(axis=(0, 1))])


def hsv_hist(img, bins=8):
    """Histogramme normalise de H, S, V (8 classes chacun) -> 24 valeurs."""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    ranges = [180, 256, 256]  # H va de 0 a 179 dans OpenCV
    feats = []
    for c in range(3):
        h = cv2.calcHist([hsv], [c], None, [bins], [0, ranges[c]]).flatten()
        feats.append(h / (h.sum() + 1e-9))
    return np.concatenate(feats)


def glcm_feats(gray):
    """Texture GLCM (distance 1, 4 angles) : 5 proprietes moyennees -> 5 valeurs."""
    glcm = graycomatrix(gray, distances=[1],
                        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                        levels=256, symmetric=True, normed=True)
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]
    return np.array([graycoprops(glcm, p).mean() for p in props])


def lbp_hist(gray, P=8, R=1):
    """Histogramme LBP 'uniform' normalise -> P+2 = 10 valeurs."""
    lbp = local_binary_pattern(gray, P, R, method="uniform")
    h, _ = np.histogram(lbp.ravel(), bins=P + 2, range=(0, P + 2))
    return h / (h.sum() + 1e-9)


def hu_moments(gray):
    """7 moments de Hu (echelle logarithmique) -> 7 valeurs."""
    hu = cv2.HuMoments(cv2.moments(gray)).flatten()
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)


def extract_features(img):
    """Vecteur de caracteristiques d'une image RGB. Taille totale : 52."""
    img = _to_uint8(img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return np.concatenate([
        rgb_stats(img),     # 6
        hsv_hist(img),      # 24
        glcm_feats(gray),   # 5
        lbp_hist(gray),     # 10
        hu_moments(gray),   # 7
    ]).astype(np.float64)


def extract_matrix(images):
    """Matrice (n_images, 52) : une ligne = une image, une colonne = une caracteristique."""
    return np.array([extract_features(im) for im in images])


## Étape A — Préparer et explorer les images

In [ ]:
import os, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from features import CLASSES, IMG_SIZE

os.makedirs("captures", exist_ok=True)

# A1 - Chargement des images : obj_car -> Voiture (0), obj_ship -> Bateau (1)
DATA_DIR = "DB2C"                     # <- chemin du dossier DB2C (à côté du notebook)
DOSSIERS = {"obj_car": 0, "obj_ship": 1}
EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

def charger_dossier(chemin):
    images = []
    for racine, _, fichiers in os.walk(chemin):
        for nom in sorted(fichiers):
            if nom.lower().endswith(EXTENSIONS):
                data = np.fromfile(os.path.join(racine, nom), dtype=np.uint8)   # lecture sûre sous Windows
                img = cv2.imdecode(data, cv2.IMREAD_COLOR)
                if img is None:
                    print("Image illisible ignorée :", nom); continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                images.append(img)
    return images

X_list, y_list = [], []
for dossier, etiquette in DOSSIERS.items():
    imgs = charger_dossier(os.path.join(DATA_DIR, dossier))
    print(f"{dossier} : {len(imgs)} images -> classe {etiquette} ({CLASSES[etiquette]})")
    X_list += imgs; y_list += [etiquette] * len(imgs)

X_all = np.array(X_list)
y_all = np.array(y_list)
print("Avant normalisation :", X_all.dtype, X_all.min(), X_all.max(), X_all.shape)
X_all = X_all.astype("float32") / 255.0          # normalisation des pixels dans [0, 1]
print("Après normalisation :", X_all.dtype, X_all.min(), X_all.max())

In [ ]:
# A2 - Séparer apprentissage (80 %) et test (20 %), en gardant la même proportion de chaque classe
# Classe positive : Bateau (1) ; classe négative : Voiture (0)
X_train_img, X_test_img, y_train_bin, y_test_bin = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)

In [ ]:
# A3 - Tailles et exemples
print("Apprentissage :", X_train_img.shape, "| étiquettes :", y_train_bin.shape)
print("Test          :", X_test_img.shape,  "| étiquettes :", y_test_bin.shape)
for name, y in [("train", y_train_bin), ("test", y_test_bin)]:
    print(f"{name} -> {CLASSES[0]} : {(y==0).sum()} | {CLASSES[1]} : {(y==1).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for lab, ax in enumerate(axes):
    ax.imshow(X_train_img[np.where(y_train_bin == lab)[0][0]])
    ax.set_title(CLASSES[lab]); ax.axis("off")
plt.tight_layout(); plt.savefig("captures/A3_exemples.png", dpi=150); plt.show()

## Étape B — Extraction des caractéristiques
RGB (moyenne/écart-type), histogramme HSV, GLCM, LBP, moments de Hu → **52 caractéristiques** par image.

In [ ]:
# B1 / B2 - Matrices de caractéristiques
from features import extract_matrix

t0 = time.time()
X_train = extract_matrix(X_train_img)
X_test  = extract_matrix(X_test_img)
print(f"Extraction terminée en {time.time()-t0:.1f} s")

print("X_train :", X_train.shape, "  (lignes = images, colonnes = caractéristiques)")
print("X_test  :", X_test.shape)
print("Détail des 52 colonnes : RGB=6 | HSV=24 | GLCM=5 | LBP=10 | Hu=7")
print("Exemple, vecteur de la 1ère image d'apprentissage :")
print(np.round(X_train[0], 3))

## Étape C — Apprentissage et prédiction (SVM, noyau RBF)

In [ ]:
# C1 - Entraînement du SVM RBF (avec standardisation des caractéristiques)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_rbf = make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42))
t0 = time.time()
svm_rbf.fit(X_train, y_train_bin)
print(f"SVM RBF entraîné en {time.time()-t0:.1f} s")
print("Accuracy sur l'apprentissage :", round(svm_rbf.score(X_train, y_train_bin), 4))

In [ ]:
# C2 - Prédire UNE image du test et comparer avec la vraie étiquette
idx = 0          # <- changez cet indice (0 à len(X_test)-1) pour tester une autre image
pred = svm_rbf.predict(X_test[idx].reshape(1, -1))[0]
vraie = y_test_bin[idx]

plt.figure(figsize=(3, 3.4))
plt.imshow(X_test_img[idx]); plt.axis("off")
plt.title(f"Prédite : {CLASSES[pred]}\nVraie : {CLASSES[vraie]}")
plt.savefig("captures/C2_prediction.png", dpi=150, bbox_inches="tight"); plt.show()
print("Prédiction correcte" if pred == vraie else "Prédiction ERRONÉE")

## Étape D — Évaluation du modèle
**Classe positive choisie : Bateau (1)**. Classe négative : Voiture (0).

In [ ]:
# D1 - Matrice de confusion
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred = svm_rbf.predict(X_test)
cm = confusion_matrix(y_test_bin, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print("Matrice de confusion (lignes = vrai, colonnes = prédit) :\n", cm)
print(f"VN (Voiture bien classée)    = {tn}")
print(f"FP (Voiture prédite Bateau)  = {fp}")
print(f"FN (Bateau prédit Voiture)  = {fn}")
print(f"VP (Bateau bien classé)     = {tp}")

plt.figure(figsize=(4.5, 3.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Classe prédite"); plt.ylabel("Classe réelle"); plt.title("SVM RBF - Matrice de confusion")
plt.tight_layout(); plt.savefig("captures/D1_matrice_confusion.png", dpi=150); plt.show()

In [ ]:
# D2 - Rapport de classification
print("Accuracy globale :", round(accuracy_score(y_test_bin, y_pred), 4))
print(classification_report(y_test_bin, y_pred, target_names=CLASSES, digits=4))

## Étape E — Étude comparative des 6 algorithmes

In [ ]:
# E1 - Même données, mêmes caractéristiques, 6 modèles
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

modeles = {
    "SVM RBF":        make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42)),
    "SVM linéaire":   make_pipeline(StandardScaler(), SVC(kernel="linear", random_state=42)),
    "SVM sigmoïde":   make_pipeline(StandardScaler(), SVC(kernel="sigmoid", random_state=42)),
    "KNN (k=3)":      make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "KNN (k=5)":      make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Arbre de décision": DecisionTreeClassifier(random_state=42),
}
lignes = []
for nom, m in modeles.items():
    t0 = time.time()
    m.fit(X_train, y_train_bin)
    acc = accuracy_score(y_test_bin, m.predict(X_test))
    lignes.append({"Modèle": nom, "Accuracy (test)": round(acc, 4), "Temps (s)": round(time.time()-t0, 1)})
    print(f"{nom:20s} accuracy = {acc:.4f}")

tableau = pd.DataFrame(lignes).sort_values("Accuracy (test)", ascending=False).reset_index(drop=True)
tableau.to_csv("captures/E1_tableau_comparatif.csv", index=False)
display(tableau)

In [ ]:
# Graphique comparatif (facultatif mais utile pour le rapport)
plt.figure(figsize=(7, 3.6))
sns.barplot(data=tableau, x="Accuracy (test)", y="Modèle", color="steelblue")
plt.xlim(0.5, 1); plt.title("Comparaison des algorithmes (accuracy sur le test)")
plt.tight_layout(); plt.savefig("captures/E1_comparaison.png", dpi=150); plt.show()

## Étape F — Page web de test (demandée par le professeur)
Cette étape sauvegarde le modèle SVM RBF, crée `app.py` et `requirements.txt`, puis lance la page web.
Le modèle de la page web est le même SVM RBF, avec `probability=True` pour afficher un pourcentage de confiance.

In [ ]:
# F1 - Entraîner et sauvegarder le modèle pour la page web
import joblib, sklearn
modele_web = make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, random_state=42))
modele_web.fit(X_train, y_train_bin)
print("Accuracy du modèle web sur le test :", round(modele_web.score(X_test, y_test_bin), 4))
joblib.dump(modele_web, "model_svm_rbf.joblib")

import cv2, skimage
with open("requirements.txt", "w") as f:
    f.write(f"gradio\nnumpy\nopencv-python-headless\nscikit-image=={skimage.__version__}\n"
            f"scikit-learn=={sklearn.__version__}\njoblib\n")
print(open("requirements.txt").read())

In [ ]:
%%writefile app.py
import gradio as gr
import joblib
import numpy as np
import cv2
from features import extract_features, CLASSES, IMG_SIZE

model = joblib.load("model_svm_rbf.joblib")

def classifier(image):
    if image is None:
        return None
    # même taille que pour l'entraînement
    petite = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    x = extract_features(petite).reshape(1, -1)
    p = model.predict_proba(x)[0]
    return {CLASSES[i]: float(p[i]) for i in range(len(CLASSES))}

demo = gr.Interface(
    fn=classifier,
    inputs=gr.Image(type="numpy", image_mode="RGB", label="Importer une image (voiture ou bateau)"),
    outputs=gr.Label(num_top_classes=2, label="Classe prédite"),
    title="Classification d'images : Voiture vs Bateau",
    description=("Modèle SVM (noyau RBF) entraîné sur la base DB2C (voitures et bateaux) avec des caractéristiques extraites "
                 "(RGB, HSV, GLCM, LBP, moments de Hu). Le modèle ne connaît que 2 classes : "
                 "toute autre image sera forcément classée Voiture ou Bateau."),
)

if __name__ == "__main__":
    demo.launch()


In [ ]:
# F2 - Lancer la page web (lien public temporaire, valable tant que le notebook tourne)
import app
app.demo.launch(share=True, debug=False)

In [ ]:
# F3 - Télécharger les fichiers nécessaires pour l'hébergement permanent
try:
    from google.colab import files
    for f in ["app.py", "features.py", "model_svm_rbf.joblib", "requirements.txt"]:
        files.download(f)
except ImportError:
    print("Hors Colab : les 4 fichiers sont déjà dans le dossier courant.")